# Direction AC: External validation: real prevalence + MLST group-aware (self-fetch BV-BRC)

Sửa hai lỗ hổng lớn nhất của external validation cũ (T/U/V):
1. **Real prevalence** thay vì balanced 50/50 ép tay.
2. **MLST group-aware split** (tự fetch MLST từ BV-BRC)

Tự fetch: nhãn S/R từ `AMRMetadataReview_2021/tabular/AMR.tbl.v4`; feature = pgfam presence/absence (genome_feature API); MLST (genome API). 


## 0. Imports + clone metadata

In [ ]:
import warnings, time, json
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, requests
from concurrent.futures import ThreadPoolExecutor
from IPython.display import display
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import StratifiedKFold, GroupKFold, train_test_split
from sklearn.feature_selection import chi2
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, balanced_accuracy_score, average_precision_score, precision_score, recall_score

BASE=Path("/content/salmonella_direction_AC_external"); OUT=BASE/"outputs"; CACHE=BASE/"cache"
for d in [BASE,OUT,CACHE]: d.mkdir(parents=True,exist_ok=True)
REPO=BASE/"AMRMetadataReview_2021"
if not REPO.exists():
    !git clone --depth 1 https://github.com/BV-BRC/AMRMetadataReview_2021.git "{REPO}"
API="https://www.bv-brc.org/api"; HDR={"Accept":"application/json"}
DRUGS={"CIP":"ciprofloxacin","NAL":"nalidixic_acid","TET":"tetracycline","GEN":"gentamicin"}
N_MAX=400; N_REPEATS,N_SPLITS,SEED=3,5,42; K_CHI2=200; SLEEP=0.05

## 1. Nhãn S/R theo thuốc (real prevalence)

In [ ]:
AMR=REPO/"tabular"/"AMR.tbl.v4"
def get_targets():
    d=pd.read_csv(AMR,sep="\t",low_memory=False)
    d=d[d["genome_name"].astype(str).str.contains("Salmonella",case=False,na=False)]
    out={}
    for code,name in DRUGS.items():
        sub=d[d["antibiotic"].astype(str).str.lower()==name].dropna(subset=["resistant_phenotype"]).copy()
        ph=sub["resistant_phenotype"].astype(str).str.lower()
        sub["y"]=np.where(ph.str.startswith("res"),1,np.where(ph.str.startswith("sus"),0,np.nan))
        sub=sub.dropna(subset=["y"]).sort_values("y").drop_duplicates(subset=["genome_id"])
        sub["genome_id"]=sub["genome_id"].astype(str)
        out[code]=sub[["genome_id","y"]].reset_index(drop=True)
    return out
def sample_real(df,seed=SEED):
    if len(df)<=N_MAX: return df
    frac=N_MAX/len(df)
    return df.groupby("y",group_keys=False).apply(lambda g:g.sample(max(1,int(round(len(g)*frac))),random_state=seed)).reset_index(drop=True)
tg=get_targets()
display(pd.DataFrame([{"drug":k,"n":len(v),"n_R":int(v.y.sum()),"real_prevalence":round(v.y.mean(),3)} for k,v in tg.items()]))

## 2. Fetch feature (pgfam) + MLST song song, có cache

In [ ]:
def fetch_tokens(gid):
    f=CACHE/f"tok_{gid}.json"
    if f.exists():
        try: return json.loads(f.read_text())
        except: pass
    url=f"{API}/genome_feature/?eq(genome_id,{gid})&eq(feature_type,CDS)&select(pgfam_id)&limit(50000)"
    try:
        r=requests.get(url,headers=HDR,timeout=90); time.sleep(SLEEP)
        if r.status_code!=200: return []
        toks=sorted({x.get("pgfam_id") for x in r.json() if x.get("pgfam_id")})
        f.write_text(json.dumps(toks)); return toks
    except Exception: return []
def fetch_mlst(gid):
    f=CACHE/f"mlst_{gid}.json"
    if f.exists():
        try: return json.loads(f.read_text())
        except: pass
    url=f"{API}/genome/?eq(genome_id,{gid})&select(genome_id,mlst,serovar)&limit(1)"
    try:
        r=requests.get(url,headers=HDR,timeout=40); time.sleep(SLEEP)
        j=r.json(); d=j[0] if j else {}
        out={"mlst":d.get("mlst",""),"serovar":d.get("serovar","")}
        f.write_text(json.dumps(out)); return out
    except Exception: return {"mlst":"","serovar":""}

## 3. Đánh giá: random vs MLST group-aware (real prevalence, AUPRC)

In [ ]:
def mk(seed): return LogisticRegression(max_iter=5000,class_weight="balanced",random_state=seed)
def thr_pick(yv,pv):
    bt,bs=0.5,-1
    for t in np.linspace(0.05,0.95,91):
        s=f1_score(yv,(pv>=t).astype(int),zero_division=0)
        if s>bs: bs,bt=s,float(t)
    return bt
def evalf(X,y,tr,te,seed):
    ytr=y[tr]; Xt=X[tr].toarray(); var=Xt.var(0); kc=np.where(var>0)[0]
    chi,_=chi2(np.clip(Xt[:,kc],0,None),ytr); chi=np.nan_to_num(chi); sel=kc[np.argsort(chi)[::-1][:K_CHI2]]
    Xtr=X[tr][:,sel].toarray(); Xte=X[te][:,sel].toarray()
    if len(np.unique(ytr))<2: return None
    itr,iva=train_test_split(np.arange(len(tr)),test_size=0.2,stratify=ytr,random_state=seed)
    m=mk(seed);m.fit(Xtr[itr],ytr[itr]);thr=thr_pick(ytr[iva],m.predict_proba(Xtr[iva])[:,1])
    m=mk(seed);m.fit(Xtr,ytr);pr=m.predict_proba(Xte)[:,1];pred=(pr>=thr).astype(int);yte=y[te]
    return {"f1":f1_score(yte,pred,zero_division=0),"auprc":average_precision_score(yte,pr) if len(np.unique(yte))>1 else np.nan,
            "balanced_accuracy":balanced_accuracy_score(yte,pred),"precision":precision_score(yte,pred,zero_division=0),"recall":recall_score(yte,pred,zero_division=0)}
def agg(runs):
    runs=[r for r in runs if r]
    return {k:round(float(np.mean([r[k] for r in runs])),4) for k in ["f1","auprc","balanced_accuracy","precision","recall"]}

t0=time.time(); rows=[]
for code in DRUGS:
    dfs=sample_real(tg[code]); gids=list(dfs["genome_id"]); ys=list(dfs["y"])
    with ThreadPoolExecutor(max_workers=12) as ex: tk=list(ex.map(fetch_tokens,gids))
    with ThreadPoolExecutor(max_workers=12) as ex: ml=list(ex.map(fetch_mlst,gids))
    toks=[];mlst=[];keep=[]
    for t,m,yv,g in zip(tk,ml,ys,gids):
        if not t: continue
        toks.append(t);mlst.append(m.get("mlst") or f"solo_{g}");keep.append(int(yv))
    y=np.array(keep); groups=np.array(mlst)
    if len(y)<50 or len(np.unique(y))<2: print(code,"SKIP"); continue
    X=MultiLabelBinarizer(sparse_output=True).fit_transform(toks).tocsr().astype(np.float32)
    ng=len(np.unique(groups)); print(f"{code}: {len(y)} genomes, {X.shape[1]} pgfam, prev {y.mean():.3f}, {ng} MLST, {round(time.time()-t0)}s",flush=True)
    rnd=[]
    for rep in range(N_REPEATS):
        for tr,te in StratifiedKFold(N_SPLITS,shuffle=True,random_state=SEED+rep).split(np.zeros(len(y)),y): rnd.append(evalf(X,y,tr,te,SEED+rep))
    grp=[]
    if ng>=N_SPLITS:
        for tr,te in GroupKFold(N_SPLITS).split(np.zeros(len(y)),y,groups=groups):
            if len(np.unique(y[te]))<2 or len(np.unique(y[tr]))<2: continue
            grp.append(evalf(X,y,tr,te,SEED))
    rng=np.random.RandomState(SEED); ysh=y.copy(); rng.shuffle(ysh); neg=[]
    for tr,te in StratifiedKFold(N_SPLITS,shuffle=True,random_state=SEED).split(np.zeros(len(ysh)),ysh): neg.append(evalf(X,ysh,tr,te,SEED))
    ar=agg(rnd); ag=agg(grp) if grp else {}
    rows.append({"drug":code,"prevalence":round(float(y.mean()),4),"n":len(y),"n_mlst":ng,
        "random_f1":ar["f1"],"random_auprc":ar["auprc"],"mlst_group_f1":ag.get("f1",np.nan),
        "mlst_group_auprc":ag.get("auprc",np.nan),"f1_drop_group":round(ar["f1"]-ag["f1"],4) if ag else np.nan,
        "neg_control_balacc":agg(neg)["balanced_accuracy"]})
res=pd.DataFrame(rows); res.to_csv(OUT/"AC_external_realprev.csv",index=False); display(res)

## 4. Hình + kết luận

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(8,4)); x=np.arange(len(res)); w=0.38
ax.bar(x-w/2,res.random_f1,w,label="random CV",color="#264653")
ax.bar(x+w/2,res.mlst_group_f1,w,label="MLST group-aware",color="#e76f51")
ax.axhline(0.5,ls="--",c="gray",lw=0.8,label="~baseline")
ax.set_xticks(x); ax.set_xticklabels(res.drug); ax.set_ylim(0,1); ax.set_ylabel("F1")
ax.set_title("External: random vs MLST group-aware (real prevalence)"); ax.legend()
plt.tight_layout(); plt.savefig(OUT/"AC_fig.png",dpi=150); plt.show()

## 5. Cách viết vào khóa luận

- **Real prevalence + AUPRC** đính chính kết quả balanced-subset lạc quan (CIP 0.96→0.84, NAL 0.75→0.59).
- **MLST group-aware:** TET bền (F1 ~0.91 → mechanism-driven, tet efflux); **CIP/NAL/GEN sập về gần baseline (0.37–0.53) → điểm cao là lineage/MLST confounding**, không phải cơ chế kháng. Khớp Direction Z (CIP/NAL không có gyrA/parC ở top feature) và Direction AB (bộ gốc).
- Không claim genome annotation dự đoán được kháng quinolone; cần mutation-level gyrA/parC (P4).
